# Algorithm 4.1 — Orbital elements from the state vector

**Goal:** given a satellite's position $\mathbf{r}$ and velocity $\mathbf{v}$ in the geocentric-equatorial frame, recover the six **classical orbital elements** that describe the orbit.

**Code (answer key):** [`coe_from_sv`](../curtis/algorithms/alg_4_1_coe_from_sv.py) · **Book:** §4.4, Algorithm 4.1 · **Worked example:** 4.3

## Read first

| Symbol | Name | Equation |
|---|---|---|
| $h$ | specific angular momentum | $\mathbf{h}=\mathbf{r}\times\mathbf{v}$ |
| $i$ | inclination | 4.7 |
| $\mathbf{N}$ | node line | 4.8 |
| $\Omega$ | right ascension of the ascending node (RAAN) | 4.9 |
| $\mathbf{e}$ | eccentricity vector | 4.10 |
| $\omega$ | argument of perigee | 4.12 |
| $\theta$ | true anomaly | 4.13 |
| $a$ | semimajor axis | 2.61 |

The whole algorithm is **vector algebra**: every element falls out of two vectors you build from the state, $\mathbf{h}$ and $\mathbf{e}$, plus the node line $\mathbf{N}$.

## The picture (why these vectors)

An orbit needs **6 numbers**: the state $(\mathbf{r},\mathbf{v})$ is 6 numbers, so the elements are just a *change of coordinates* into ones with geometric meaning.

- **$\mathbf{h} = \mathbf{r}\times\mathbf{v}$** is perpendicular to the orbital plane. Its *direction* pins the plane; its *magnitude* sets the orbit's size and shape (with $e$). Two angles describe the plane's tilt: **$i$** (tip from the equator) and **$\Omega$** (where it crosses the equator going north).
- **$\mathbf{N} = \hat{\mathbf{K}}\times\mathbf{h}$** lies *in* the equatorial plane and points at the **ascending node**. It is the reference direction for $\Omega$ and $\omega$.
- **$\mathbf{e}$** points from the focus toward **perigee**, with $|\mathbf{e}|=e$. It is the reference direction for $\theta$.

So: $\mathbf{h}$ gives the plane, $\mathbf{N}$ gives a line in it, $\mathbf{e}$ gives perigee, and $\theta$ is the satellite's angle past perigee.

The function returns `[h, e, RA, incl, w, TA, a]` (angles in **radians**, $h$ in km²/s, $a$ in km).

In [1]:
import numpy as np
import orbital_elements

## See it — the frame, the origin, the angles

The orbit lives in the **geocentric-equatorial frame** (ECI). Rotate the figure above to see it from any side.

- **Origin $O$ = Earth's centre.** Your $\mathbf{r}$ and $\mathbf{v}$ are measured from here.
- **$\hat{\mathbf{I}}$ → vernal equinox** — a *fixed* direction in space that every angle is referenced to. The frame does **not** spin with the Earth.
- **$\hat{\mathbf{K}}$ → Earth's rotation axis** (north); **$\hat{\mathbf{J}}$** completes the right-handed set. The `[0,0,1]` in your step 4 *is* $\hat{\mathbf{K}}$.
- **Fundamental plane = the equator** ($\hat{\mathbf{I}}\,\hat{\mathbf{J}}$).

The four angles form a chain — $\hat{\mathbf{I}}\xrightarrow{\;\Omega\;}$ node $\xrightarrow{\;\omega\;}$ perigee $\xrightarrow{\;\theta\;}$ satellite — which is why each builds on the one before:

| angle | lives in | measured from → to |
|---|---|---|
| $\Omega$ | equatorial plane | $\hat{\mathbf{I}}$ → ascending node |
| $i$ | between the planes | tilt at the node |
| $\omega$ | orbit plane | node → perigee |
| $\theta$ | orbit plane | perigee → satellite |

$\mathbf{N}$ is the **hinge**: it belongs to *both* planes, so it's where you hand off from the equator ($\Omega$) up into the orbit plane ($\omega, \theta$).

In [7]:
# If you want to see cross-sections of the image, then uncomment and run the following:
# for f in ("frame", "raan", "inclination", "argp", "ta"):
#     orbital_elements.plot_orbital_elements(focus=f).show()

In [8]:
# This one show the frames of refrences and all the orbital elements together
orbital_elements.plot_orbital_elements(focus="all").show()

## Where these equations come from

Two facts generate **all six** elements: (1) gravity is a *central* force, so angular momentum is conserved; (2) integrating the equation of motion once produces a *second* conserved vector, the eccentricity vector. Everything else is projecting those two vectors onto the frame.

### The one rule behind every angle
The angle between two directions $\mathbf{a},\mathbf{b}$ is $\cos^{-1}\!\big(\tfrac{\mathbf{a}\cdot\mathbf{b}}{|\mathbf{a}|\,|\mathbf{b}|}\big)$. Each of $i,\Omega,\omega,\theta$ is *just this formula* with a different pair of vectors — that is why steps 3, 5, 7, 8 all look identical. The only extra work is the half-plane (quadrant) fix.

### 1 · Angular momentum is conserved ⇒ the orbit is flat
The two-body equation of motion is $\ddot{\mathbf{r}} = -\dfrac{\mu}{r^3}\mathbf{r}$. With $\mathbf{h}=\mathbf{r}\times\mathbf{v}$,
$$\dot{\mathbf{h}}=\dot{\mathbf{r}}\times\mathbf{v}+\mathbf{r}\times\dot{\mathbf{v}}=\underbrace{\mathbf{v}\times\mathbf{v}}_{0}+\mathbf{r}\times\Big(\!-\tfrac{\mu}{r^3}\mathbf{r}\Big)=\mathbf{0}.$$
So $\mathbf{h}$ is **constant** in size and direction. Since $\mathbf{h}\perp\mathbf{r}$ always, $\mathbf{r}$ never leaves the plane $\perp\mathbf{h}$ — the orbit is planar. That is why step 2 builds $\mathbf{h}$ first: it *is* the orbit plane.

### 2 · Inclination and node line are projections of $\mathbf{h}$
The frame's axis unit vectors are just the columns of the identity matrix:
$$\hat{\mathbf{I}}=[1,0,0],\qquad \hat{\mathbf{J}}=[0,1,0],\qquad \hat{\mathbf{K}}=[0,0,1].$$
So dotting any vector with one of them simply **picks out that component** (this is the `[0,0,1]` in your step 4).

The satellite's orbit plane's normal is $\hat{\mathbf{h}}=\mathbf{h}/h$; the equatorial plane's normal is $\hat{\mathbf{K}}$. The angle between two planes equals the angle between their normals, and for unit vectors the dot product *is* the cosine:
$$\cos i=\hat{\mathbf{h}}\cdot\hat{\mathbf{K}}=\frac{h_x}{h}(0)+\frac{h_y}{h}(0)+\frac{h_z}{h}(1)=\frac{h_z}{h}\quad\text{(step 3).}$$
The **line of nodes** is where the two planes meet; a direction lying in both planes must be perpendicular to both normals, i.e. their cross product, $\mathbf{N}=\hat{\mathbf{K}}\times\mathbf{h}$ (step 4). The same component-picking trick gives $\cos\Omega=\hat{\mathbf{N}}\cdot\hat{\mathbf{I}}=N_x/n$ (step 5).

### 3 · The eccentricity vector — the heart of it (Eq 4.10)
Cross the equation of motion with the constant $\mathbf{h}$. The left side becomes a total derivative because $\mathbf{h}$ is constant, and the right side simplifies (using $\mathbf{h}=\mathbf{r}\times\dot{\mathbf{r}}$ and the triple product) to $\mu\,\tfrac{d}{dt}(\mathbf{r}/r)$:
$$\frac{d}{dt}\big(\mathbf{v}\times\mathbf{h}\big)=\mu\,\frac{d}{dt}\!\Big(\frac{\mathbf{r}}{r}\Big).$$
Integrate once. The constant of integration is a new fixed vector $\mathbf{e}$:
$$\mathbf{v}\times\mathbf{h}=\mu\Big(\frac{\mathbf{r}}{r}+\mathbf{e}\Big).$$
Solve for $\mathbf{e}$ and expand $\mathbf{v}\times\mathbf{h}=\mathbf{v}\times(\mathbf{r}\times\mathbf{v})=v^2\mathbf{r}-(\mathbf{r}\cdot\mathbf{v})\mathbf{v}$ (the "$BAC-CAB$" identity):
$$\boxed{\;\mathbf{e}=\frac{1}{\mu}\Big[\big(v^2-\tfrac{\mu}{r}\big)\mathbf{r}-(\mathbf{r}\cdot\mathbf{v})\,\mathbf{v}\Big]\;}$$
— exactly step 6, since $(\mathbf{r}\cdot\mathbf{v})=r\,v_r$. Like $\mathbf{h}$, $\mathbf{e}$ is **constant**: a fixed arrow lying in the orbit plane.

### 4 · Why $\mathbf{e}$ points at perigee, and where $a$ comes from
Dot $\mathbf{v}\times\mathbf{h}=\mu(\mathbf{r}/r+\mathbf{e})$ with $\mathbf{r}$ and use $\mathbf{r}\cdot(\mathbf{v}\times\mathbf{h})=h^2$:
$$h^2=\mu\,(r+\mathbf{e}\cdot\mathbf{r})=\mu r\,(1+e\cos\theta),\qquad \theta\equiv\angle(\mathbf{e},\mathbf{r}),$$
which rearranges to the **orbit equation** (a conic section):
$$r=\frac{h^2/\mu}{1+e\cos\theta}.$$
$r$ is smallest when $\cos\theta=1$, i.e. when $\mathbf{r}$ aligns with $\mathbf{e}$ — so **$\mathbf{e}$ points to perigee** and $|\mathbf{e}|=e$ is the eccentricity. That is why $\omega$ is measured node $\to\mathbf{e}$ (step 7) and $\theta$ is measured $\mathbf{e}\to\mathbf{r}$ (step 8).

The numerator is the **semi-latus rectum** $p=h^2/\mu$. For any conic $p=a(1-e^2)$, hence
$$a=\frac{p}{1-e^2}=\frac{h^2/\mu}{1-e^2}\quad\text{(step 9).}$$
Equivalently the energy $\varepsilon=\tfrac{v^2}{2}-\tfrac{\mu}{r}=-\tfrac{\mu}{2a}$ gives the same $a$; for a hyperbola $e>1$ so $a<0$.

### The quadrant fixes, explained
$\cos^{-1}x$ only returns $x \in [0,180°]$, so it cannot distinguish a direction from its mirror image. Each test picks the correct half:
- $N_y<0$: ascending node lies below the $\hat{\mathbf{I}}$ axis ⇒ $\Omega>180°$.
- $e_z<0$: perigee sits below the equator ⇒ $\omega>180°$.
- $v_r=\mathbf{r}\cdot\mathbf{v}<0$: the satellite is falling back toward perigee (past apogee) ⇒ $\theta>180°$.

## Step by step (in code order)

Use a small `eps = 1e-10` to decide when $e$ or $n$ is "zero".

**1. Magnitudes and radial speed.** $\;r=\lVert\mathbf{r}\rVert,\ v=\lVert\mathbf{v}\rVert,\ v_r=\dfrac{\mathbf{r}\cdot\mathbf{v}}{r}$
The **sign of $v_r$** is the quadrant trick used below: $v_r>0$ ⇒ climbing away from perigee ($0<\theta<180°$); $v_r<0$ ⇒ falling toward it.

**2. Angular momentum.** $\;\mathbf{h}=\mathbf{r}\times\mathbf{v},\quad h=\lVert\mathbf{h}\rVert$

**3. Inclination (Eq 4.7).** $\;i=\cos^{-1}\!\left(h_z/h\right)\in[0,180°]$ — no quadrant ambiguity.

**4. Node line (Eq 4.8).** $\;\mathbf{N}=[0,0,1]\times\mathbf{h},\quad n=\lVert\mathbf{N}\rVert$

**5. RAAN (Eq 4.9).** $\;\Omega=\cos^{-1}\!\left(N_x/n\right)$, and if $N_y<0:\ \Omega=2\pi-\Omega$.

**6. Eccentricity vector (Eq 4.10).** $\;\mathbf{e}=\dfrac1\mu\!\left[(v^2-\mu/r)\,\mathbf{r}-r\,v_r\,\mathbf{v}\right],\quad e=\lVert\mathbf{e}\rVert$

**7. Argument of perigee (Eq 4.12).** $\;\omega=\cos^{-1}\!\left(\dfrac{\mathbf{N}\cdot\mathbf{e}}{n\,e}\right)$, and if $e_z<0:\ \omega=2\pi-\omega$.

**8. True anomaly (Eq 4.13).** $\;\theta=\cos^{-1}\!\left(\dfrac{\mathbf{e}\cdot\mathbf{r}}{e\,r}\right)$, and if $v_r<0:\ \theta=2\pi-\theta$.

**9. Semimajor axis (Eq 2.61).** $\;a=\dfrac{h^2}{\mu}\,\dfrac{1}{1-e^2}$ (hyperbola ⇒ $a<0$ automatically).

**↓ Now type your implementation below.** Build the normal path (steps 1–9) first; add the `if n != 0 / if e > eps` singular-case guards afterwards. The module linked above is your answer key — peek only *after* you've tried.

In [3]:
def norm(n):
    return np.linalg.norm(n)
    
def coe_from_sv(R, V, mu):
    """Classical orbital elements [h, e, RA, incl, w, TA, a] from state (R, V)."""
    R = np.asarray(R, dtype=float)
    V = np.asarray(V, dtype=float)
    eps = 1.0e-10

    # 1. Magnitudes r, v and radial speed  vr = (R . V) / r
    r = norm(R)
    vr = np.divide(np.dot(R, V),r)
    # 2. Angular momentum  H = R x V,  h = |H|
    H = np.cross(R,V)
    h = norm(H)
    # 3. Inclination (Eq 4.7):  incl = acos(H_z / h)
    incl = np.arccos(H[2]/h)
    # 4. Node line (Eq 4.8):  N = [0,0,1] x H,  n = |N|
    N = np.cross([0,0,1],H)
    n = norm(N)
    # 5. RAAN (Eq 4.9):  RA = acos(N_x / n); if N_y < 0: RA = 2*pi - RA
    #    (singular: if n == 0, set RA = 0)
    if n < eps: RA = 0
    else: 
        RA = np.arccos(N[0]/n)
        if N[1]<0: RA = 2*np.pi - RA
    # 6. Eccentricity vector (Eq 4.10):  E = (1/mu)[(v^2 - mu/r) R - r*vr V],  e = |E|
    E = ((norm(V)**2 - mu/r)*R - r*vr*V)/mu
    e = norm(E)
    # 7. Argument of perigee (Eq 4.12):  w = acos(N . E / (n e)); if E_z < 0: w = 2*pi - w
    #    (singular: if e ~ 0, set w = 0)
    if e < eps or n < eps: w = 0
    else: 
        w = np.arccos(np.dot(N,E)/(n*e))
        if E[2]<0: w = 2*np.pi - w
    # 8. True anomaly (Eq 4.13):  TA = acos(E . R / (e r)); if vr < 0: TA = 2*pi - TA
    #    (singular: if e ~ 0, use argument of latitude / true longitude)
    if e < eps: 
        cp = np.cross(N, R)
        if cp[2] >= 0:
            TA = np.arccos(np.dot(N, R) / (n * r))
        else:
            TA = 2 * np.pi - np.arccos(np.dot(N, R) / (n * r))
    else: 
        TA = np.arccos(np.dot(E,R)/(e*r))
        if vr<0: TA = 2*np.pi - TA
    # 9. Semimajor axis (Eq 2.61):  a = h^2 / mu / (1 - e^2)
    a = h**2 / ((1-e**2)*mu)
    return np.array([h, e, RA, incl, w, TA, a])
    # raise NotImplementedError("fill in steps 1-9, then delete this line")

## Singular cases (add these guards after the normal path works)

| Condition | Meaning | Fallback |
|---|---|---|
| $n\approx 0$ | equatorial ($i=0°$ or $180°$) — node undefined | set $\Omega=0$; measure from $\hat{\mathbf{I}}$ |
| $e\approx 0$ | circular — perigee undefined | set $\omega=0$; use argument of latitude (from $\mathbf{N}$), or true longitude (from $\hat{\mathbf{I}}$) if also equatorial |

In the circular branch use $\mathbf{c}=\mathbf{N}\times\mathbf{r}$ and the sign of $c_z$ as the quadrant test in place of $v_r$.

## Verify — Example 4.3

**Input** ($\mu_\oplus=398600$):
$\;\mathbf{r}=[-6045,-3490,2500]$ km, $\;\mathbf{v}=[-3.457,6.618,2.533]$ km/s.

| $h$ | $e$ | $\Omega$ | $i$ | $\omega$ | $\theta$ | $a$ |
|---|---|---|---|---|---|---|
| 58311.7 | 0.171212 | 255.279° | 153.249° | 20.0683° | 28.4456° | 8788.10 km |

Run the cell below once your function is typed — it asserts every value against the book.

In [4]:
r  = np.array([-6045.0, -3490.0, 2500.0])
v  = np.array([-3.457, 6.618, 2.533])
mu = 398600.0

coe = coe_from_sv(r, v, mu)
h, e, RA, incl, w, TA, a = coe

print(f"h    = {h:10.5g} km^2/s   (expect 58311.7)")
print(f"e    = {e:10.6g}          (expect 0.171212)")
print(f"RA   = {np.degrees(RA):10.5g} deg      (expect 255.279)")
print(f"incl = {np.degrees(incl):10.5g} deg      (expect 153.249)")
print(f"w    = {np.degrees(w):10.5g} deg      (expect 20.0683)")
print(f"TA   = {np.degrees(TA):10.5g} deg      (expect 28.4456)")
print(f"a    = {a:10.6g} km       (expect 8788.10)")

assert abs(h - 58311.7) < 0.1
assert abs(e - 0.171212) < 1e-5
assert abs(np.degrees(RA) - 255.279) < 1e-3
assert abs(np.degrees(incl) - 153.249) < 1e-3
assert abs(np.degrees(w) - 20.0683) < 1e-3
assert abs(np.degrees(TA) - 28.4456) < 1e-3
assert abs(a - 8788.10) < 0.1
print("\nAll checks passed ✔")

h    =      58312 km^2/s   (expect 58311.7)
e    =   0.171212          (expect 0.171212)
RA   =     255.28 deg      (expect 255.279)
incl =     153.25 deg      (expect 153.249)
w    =     20.068 deg      (expect 20.0683)
TA   =     28.446 deg      (expect 28.4456)
a    =     8788.1 km       (expect 8788.10)

All checks passed ✔


## What this confirms

- The classical elements are a pure change of variables from $(\mathbf{r},\mathbf{v})$ — no integration, no time.
- Angles need an explicit half-plane test (`acos` is only half the circle); each flip ($N_y$, $e_z$, $v_r$) has a physical meaning.
- $i=153°>90°$ here ⇒ a **retrograde** orbit; cross-check that $h_z<0$.

**Next:** type [Algorithm 4.2](../curtis/algorithms/alg_4_2_sv_from_coe.py) (the inverse) and round-trip Example 4.3 ⇄ 4.5 to prove both directions agree.